# B2B SaaS Revenue Quality & Churn Early Warning OS

This notebook provides an end-to-end walkthrough of the analytical workflow and its main outputs.

It reads from `data/raw/` (generated, not tracked) and `data/processed/` — on a fresh clone, run `make data` (or the full pipeline) first. See `data/README.md`.

## 1) Business Question
Are we growing through healthy recurring revenue quality, or through fragile discount-led growth with hidden churn risk?

In [1]:
from pathlib import Path
import json
import pandas as pd

base = Path('..').resolve()
raw = base / 'data' / 'raw'
processed = base / 'data' / 'processed'
reports = base / 'reports'

summary = json.loads((reports / 'formal_validation_summary.json').read_text())
summary

{'overall_assessment': 'Technically valid. Validation controls passed without material caveats.',
 'summary': {'total_findings': 21,
  'status_counts': {'PASS': 21, 'WARN': 0, 'FAIL': 0},
  'severity_counts': {'Critical': 0,
   'High': 0,
   'Medium': 0,
   'Low': 0,
   'None': 21},
  'readiness': {'tier': 'technically valid',
   'rationale': 'All governed controls passed with no warnings or failures.'}},
 'readiness': {'tier': 'technically valid',
  'rationale': 'All governed controls passed with no warnings or failures.'},
 'readiness_scale': ['publish-blocked',
  'not committee-grade',
  'screening-grade only',
  'decision-support only',
  'analytically acceptable',
  'technically valid'],
 'confidence_by_component': [{'component': 'Raw Data Logic',
   'confidence': 'High',
   'pass': 6,
   'warn': 0,
   'fail': 0},
  {'component': 'Processed Tables',
   'confidence': 'High',
   'pass': 2,
   'warn': 0,
   'fail': 0},
  {'component': 'Feature Engineering',
   'confidence': 'High',
 

## 2) Data Footprint

In [2]:
tables = {
    'customers': pd.read_csv(raw / 'customers.csv'),
    'subscriptions': pd.read_csv(raw / 'subscriptions.csv'),
    'monthly_account_metrics': pd.read_csv(raw / 'monthly_account_metrics.csv'),
    'invoices': pd.read_csv(raw / 'invoices.csv'),
    'account_scoring_model_output': pd.read_csv(processed / 'account_scoring_model_output.csv'),
}
pd.DataFrame({k: [len(v), len(v.columns)] for k, v in tables.items()}, index=['rows', 'cols']).T

,rows,cols
customers,4500,9
subscriptions,99729,11
monthly_account_metrics,112038,13
invoices,99729,10
account_scoring_model_output,4500,26


## 3) Core KPIs

In [3]:
metrics = json.loads((reports / 'main_business_analysis_metrics.json').read_text())
sec1 = metrics['section1']
sec2 = metrics['section2']
{
    'mrr_start': sec1['mrr_start'],
    'mrr_end': sec1['mrr_end'],
    'arr_end': sec1['arr_end'],
    'latest_grr': sec2['latest_grr'],
    'latest_nrr': sec2['latest_nrr'],
    'discounted_revenue_share_latest': sec1['share_discounted_mrr_latest'],
    'at_risk_mrr_share_latest': sec1['share_at_risk_mrr_latest'],
}

{'mrr_start': 3749108.29,
 'mrr_end': 9523589.88,
 'arr_end': 114283078.56,
 'latest_grr': 0.991727,
 'latest_nrr': 0.998248,
 'discounted_revenue_share_latest': 0.159462,
 'at_risk_mrr_share_latest': 0.039501}

## 4) Scoring Snapshot

In [4]:
scores = pd.read_csv(processed / 'account_scoring_model_output.csv')
scores[['churn_risk_tier','governance_priority_tier']].value_counts().head(10)

churn_risk_tier  governance_priority_tier
Low              Low                         2627
                 Moderate                    1610
Moderate         Moderate                     172
                 High                          45
High             High                          30
                 Moderate                      11
Low              High                           4
High             Critical                       1
Name: count, dtype: int64

## 5) Forecast & Scenario Snapshot

In [5]:
pd.read_csv(processed / 'mrr_scenario_table.csv').sort_values('scenario')

,scenario,scenario_type,horizon_months,start_mrr,end_mrr,end_arr,end_realized_arr_estimate,mrr_change,mrr_growth_pct,avg_assumed_expansion_rate,avg_assumed_contraction_rate,avg_assumed_churn_rate,avg_assumed_net_new_rate,mrr_vs_base,arr_vs_base
0,base_case,base,6,9523589.88,10647476.20,1.277697e+08,1.050139e+08,1123886.32,0.1180,0.00678,0.00302,0.00621,0.02122,0.00,0.00
1,discount_discipline_improvement_case,policy-improvement,6,9523589.88,10647731.53,1.277728e+08,1.075719e+08,1124141.65,0.1180,0.00637,0.00271,0.00547,0.02058,255.33,3063.96
2,downside_case,fragile-growth,6,9523589.88,9923686.57,1.190842e+08,9.668449e+07,400096.69,0.0420,0.00542,0.00407,0.00932,0.01485,-723789.63,-8685475.56
3,improvement_case,healthy-growth,6,9523589.88,11022495.13,1.322699e+08,1.093740e+08,1498905.25,0.1574,0.00779,0.00256,0.00497,0.02440,375018.93,4500227.16
4,risk_adjusted_case,risk-adjusted,6,9523589.88,10277564.33,1.233308e+08,1.007489e+08,753974.45,0.0792,0.00677,0.00509,0.00933,0.02043,-369911.87,-4438942.44


## 6) Validation Status (Release Gate)

In [6]:
summary['summary']

{'total_findings': 21,
 'status_counts': {'PASS': 21, 'WARN': 0, 'FAIL': 0},
 'severity_counts': {'Critical': 0,
  'High': 0,
  'Medium': 0,
  'Low': 0,
  'None': 21},
 'readiness': {'tier': 'technically valid',
  'rationale': 'All governed controls passed with no warnings or failures.'}}

## 7) Artifacts
- Dashboard: `../outputs/dashboard/revenue-quality-command-center.html`
- Validation summary: `../reports/formal_validation_summary.json`